# Reciprocal U8/U16 int32 smoke test

Focused notebook for `trptq_softmax_dp4a.reciprocal_u8` and `reciprocal_u16` with raw int32 denominators. Outputs are compared against torch baselines scaled by `2**8` and `2**16`.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

In [ ]:
import math
import shutil
import sys
from pathlib import Path

import torch

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

INT32_MAX = (1 << 31) - 1
U8_SCALE = 1 << 8
U16_SCALE = 1 << 16
SOFTMAX_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "softmax"

In [ ]:
# Rebuild after editing CUDA/C++ sources.
softmax_dir = Path(SOFTMAX_EXT_DIR)
shutil.rmtree(softmax_dir / "build", ignore_errors=True)
for so_path in softmax_dir.glob("_trptq_softmax_dp4a*.so"):
    so_path.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{SOFTMAX_EXT_DIR}"

In [ ]:
import importlib
import _trptq_softmax_dp4a
import trptq_softmax_dp4a

trptq_softmax_dp4a = importlib.reload(trptq_softmax_dp4a)
print("Loaded:", trptq_softmax_dp4a.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

for name in ["recip_u8", "recip_u16"]:
    if not hasattr(_trptq_softmax_dp4a, name):
        raise RuntimeError(f"Missing CUDA export: {name}. Rebuild and restart the runtime.")
print("reciprocal exports: OK")

In [ ]:
def baseline_u8(x):
    return torch.round((1.0 / x.float()) * U8_SCALE).clamp(0, 255).to(torch.uint8)


def baseline_u16(x):
    return torch.round((1.0 / x.float()) * U16_SCALE).clamp(0, 65535).to(torch.uint16)


def sample_denominators(device):
    fixed = torch.tensor(
        [
            1, 2, 3, 4, 8, 15, 16, 17, 31, 32, 33,
            127, 128, 129, 255, 256, 257,
            1023, 1024, 1025, 65535, 65536, 65537,
            (1 << 20) - 1, 1 << 20, (1 << 20) + 1,
            (1 << 24) - 1, 1 << 24, (1 << 24) + 1,
            (1 << 30) - 1, 1 << 30, (1 << 30) + 1,
            INT32_MAX - 2, INT32_MAX - 1, INT32_MAX,
        ],
        device=device,
        dtype=torch.int64,
    )
    powers = 2 ** torch.arange(0, 31, device=device, dtype=torch.int64)
    grid = torch.logspace(0, math.log10(INT32_MAX), steps=4096, device=device)
    grid = grid.round().clamp(1, INT32_MAX).to(torch.int64)
    return torch.unique(torch.cat([fixed, powers, grid])).to(torch.int32)


def metrics(name, got, ref, scale):
    got_i = got.detach().cpu().to(torch.int64)
    ref_i = ref.detach().cpu().to(torch.int64)
    err = got_i - ref_i
    err_f = err.double() / scale
    return {
        "name": name,
        "max_q": int(err.abs().max()),
        "mean_q": err.abs().double().mean().item(),
        "max_f": err_f.abs().max().item(),
        "mean_f": err_f.abs().mean().item(),
        "rmse_f": err_f.square().mean().sqrt().item(),
    }


def print_metrics(rows):
    print(f"{'variant':<16} {'max_q':>8} {'mean_q':>10} {'max_f':>12} {'mean_f':>12} {'rmse_f':>12}")
    print("-" * 74)
    for row in rows:
        print(
            f"{row['name']:<16} {row['max_q']:8d} {row['mean_q']:10.4f} "
            f"{row['max_f']:12.4e} {row['mean_f']:12.4e} {row['rmse_f']:12.4e}"
        )

In [ ]:
assert torch.cuda.is_available(), "Select a CUDA runtime first."
device = torch.device("cuda")

x = sample_denominators(device)
u8_ref = baseline_u8(x)
u16_ref = baseline_u16(x)
u8_cuda = trptq_softmax_dp4a.reciprocal_u8(x)
u16_cuda = trptq_softmax_dp4a.reciprocal_u16(x)

one_idx = (x == 1).nonzero(as_tuple=True)[0][0]
assert int(u8_cuda[one_idx]) == 255
assert int(u16_cuda[one_idx]) == 65535

print_metrics([
    metrics("u8", u8_cuda, u8_ref, U8_SCALE),
    metrics("u16", u16_cuda, u16_ref, U16_SCALE),
])

In [ ]:
def print_detail(title, got, ref, scale):
    x_cpu = x.cpu()
    got_cpu = got.detach().cpu().to(torch.int64)
    ref_cpu = ref.detach().cpu().to(torch.int64)
    wanted = [1, 2, 3, 4, 8, 16, 32, 128, 255, 256, 257, 1024, 65535, 65536, 65537, 1 << 20, 1 << 24, 1 << 30, INT32_MAX]
    index = {int(v): i for i, v in enumerate(x_cpu.tolist())}
    print(title)
    print(f"{'x':>12} {'ref_q':>8} {'cuda_q':>8} {'err_q':>8} {'ref_f':>11} {'cuda_f':>11}")
    print("-" * 66)
    for value in wanted:
        if value not in index:
            continue
        i = index[value]
        ref_q = int(ref_cpu[i])
        got_q = int(got_cpu[i])
        print(f"{value:12d} {ref_q:8d} {got_q:8d} {got_q - ref_q:8d} {ref_q / scale:11.6f} {got_q / scale:11.6f}")


print_detail("uint8 reciprocal", u8_cuda, u8_ref, U8_SCALE)
print()
print_detail("uint16 reciprocal", u16_cuda, u16_ref, U16_SCALE)

In [ ]:
def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


x_big = torch.randint(1, INT32_MAX, (1_000_000,), device=device, dtype=torch.int32)
u8_ms = benchmark_cuda(lambda: trptq_softmax_dp4a.reciprocal_u8(x_big))
u8_ref_ms = benchmark_cuda(lambda: baseline_u8(x_big))
u16_ms = benchmark_cuda(lambda: trptq_softmax_dp4a.reciprocal_u16(x_big))
u16_ref_ms = benchmark_cuda(lambda: baseline_u16(x_big))

print(f"{'kernel':<18} {'ms':>10} {'speedup':>10}")
print("-" * 42)
print(f"{'reciprocal_u8':<18} {u8_ms:10.4f} {u8_ref_ms / u8_ms:9.2f}x")
print(f"{'torch_u8':<18} {u8_ref_ms:10.4f} {'1.00':>9}x")
print(f"{'reciprocal_u16':<18} {u16_ms:10.4f} {u16_ref_ms / u16_ms:9.2f}x")
print(f"{'torch_u16':<18} {u16_ref_ms:10.4f} {'1.00':>9}x")